# QAN Parameter Sweep — Optuna (Experiment 1, CAN-only)

Tunes the two QAN **accuracy** axes, `sigma` and `offset_magnitude`, with Optuna's
TPE (Bayesian) sampler. The search is **CAN-only**: the arena's torus ground-truth
trajectory is fed straight into the QAN via `TorchBackend.simulate`, bypassing the
Bingham filter (which is idle on the flat 2-D arena anyway). The winning point is
then validated in the *full* pipeline at the bottom.

**Design assumptions baked in (from the refactor):**
- `velocity_gains` is **not** a search axis — it is derived as `τ / offset_magnitude`
  inside the QAN, so it recalibrates automatically as Optuna varies `offset_magnitude`.
- `tau` lives **only in the CAN** (default 5). `Torus3DQAN` no longer takes a `tau`
  argument, so the objective does **not** pass one. *(If you kept `tau` as a QAN field
  instead, add `tau=5.0` to the constructor below.)*
- `kernel_alpha` (`ALPHA`) and `b` are feasibility knobs, held fixed; failed cells are
  caught, not silently averaged.

**Before scaling:** keep `N_SIDE = 21`. At n = 64 the MADE construction allocates a
dense N×N connectivity matrix (~275 GB per CAN) *before* torch takes over — it OOMs.
Raise `N_SIDE` only after the lazy-connectivity fix.

**Backend caveat:** this sweep optimizes the **torch blend** backend. If your
`PathIntegrator` drives the NumPy *additive* path (`compute_can_input`) instead, the
two mechanisms calibrate the gain differently — re-point the sweep at that backend,
or the winner may not transfer. Check `path_integration.py`.

Requires: `pip install optuna plotly`.

In [1]:
import os, sys, gc
import numpy as np
import optuna

# Adjust to your layout (mirrors notebook 4_pipeline_integration).
sys.path.insert(0, os.path.abspath("../.."))

from config import RunConfig, NetworkConfig
from experiments.arena_2d import Arena2DExperiment, Arena2DConfig
from experiments.base import BaseExperiment
from network.QAN3D import Torus3DQAN
from network.torch_backend import TorchBackend

# ---- fixed setup ---------------------------------------------------------
N_SIDE  = 10                     # raise ONLY after the lazy-connectivity fix (n=64 OOMs at construction)
SPACING = 2 * np.pi / N_SIDE     # spacing that yields exactly N_SIDE neurons / axis
ALPHA   = 1.5                    # kernel amplitude  (feasibility knob, fixed)
B       = 0.83                   # feedforward drive (feasibility knob, fixed)
T       = 300                   # shorter than the 5000-step pipeline run, for search speed
# tau is NOT set here: it lives in the CAN (default 5); gain = tau/offset is derived in the QAN.

## 1 — Trajectory (generated once, via the arena)

The trajectory depends only on the manifold metric and the experiment config —
**not** on `sigma`/`offset`/`alpha` — so we generate it once and every trial sees the
*same* input. That makes differences across trials purely parametric, not random-walk
noise.

In [2]:
base = Arena2DExperiment(
    RunConfig(network=NetworkConfig(), experiment=Arena2DConfig(n_steps=T, seed=0)),
    record=False,
)
world_pos, v_body_seq, torus_gt = base.generate_trajectory()
print("torus_gt:", torus_gt.shape, " world_pos:", world_pos.shape)
del base; gc.collect()   # the trajectory QAN is no longer needed

torus_gt: (300, 3)  world_pos: (300, 3)


40

## 2 — Objective + study

CAN-only: build the QAN with the trial's `(sigma, offset)`, run `TorchBackend.simulate`
on the fixed `torus_gt`, return the **post-transient** MADE error (same metric, same
`world_pos` normalization as the full pipeline, so the numbers are comparable).

Robustness:
- a config that NaNs / OOMs returns a large **finite** penalty rather than `inf` —
  a completed-but-bad trial keeps the TPE surrogate well-behaved *and* steers it away
  from the failing region (a propagated exception would be excluded from the model);
- a silent non-finite result is also penalized.

In [3]:
PENALTY = 1e3   # clearly worse than any valid MADE error (which is << 10)

def objective(trial):
    sg  = trial.suggest_float("sigma",            0.8,  1.7)
    off = trial.suggest_float("offset_magnitude", 0.15, 0.50)
    try:
        qan = Torus3DQAN(spacing=SPACING, sigma=sg, offset_magnitude=off,
                         alpha=ALPHA, b=B)                 # no tau: CAN default (5) is used
        decoded = TorchBackend(qan=qan).simulate(torus_gt)
        err = BaseExperiment._made_metric(decoded, torus_gt, world_pos=world_pos)
        val = float(err[200:].mean())                     # post-transient mean
        return val if np.isfinite(val) else PENALTY
    except (ValueError, RuntimeError, MemoryError) as e:
        print(f"FAIL sigma={sg:.3f} off={off:.3f}: {type(e).__name__}")
        return PENALTY
    finally:
        gc.collect()   # free the dense matrices torch already dropped, between trials

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=0),   # ~10 random startups, then TPE
)
study.optimize(objective, n_trials=2)

print("best params:", study.best_params)
print("best error :", study.best_value)

[I 2026-05-30 14:05:11,987] A new study created in memory with name: no-name-a5ad6f01-1b67-4f14-8033-af6df4308314
[I 2026-05-30 14:05:20,642] Trial 0 finished with value: 18.755541503243993 and parameters: {'sigma': 1.2939321535345922, 'offset_magnitude': 0.40031627823034677}. Best is trial 0 with value: 18.755541503243993.
[I 2026-05-30 14:05:27,510] Trial 1 finished with value: 17.83587782921516 and parameters: {'sigma': 1.3424870384644794, 'offset_magnitude': 0.3407091140489139}. Best is trial 1 with value: 17.83587782921516.


best params: {'sigma': 1.3424870384644794, 'offset_magnitude': 0.3407091140489139}
best error : 17.83587782921516


## 3 — Inspect the search

`plot_contour` is the useful one here: with two axes it shows the response surface and
the `sigma`–`offset` interaction directly. `plot_param_importances` quantifies which
axis moved the error more. (Requires `plotly`.)

In [4]:
import optuna.visualization as vis

vis.plot_optimization_history(study).show()
vis.plot_contour(study, params=["sigma", "offset_magnitude"]).show()
vis.plot_param_importances(study).show()

study.trials_dataframe().sort_values("value").head(10)

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

## 4 — Validate the winner in the *full* pipeline

The sweep was CAN-only. Now re-introduce the Bingham filter and the metres↔radians
mapping by running the real `Arena2DExperiment` at the winning params, across a few
seeds, so you know the result isn't seed luck. A discrepancy between these numbers and
the sweep's best value is informative: on the flat arena the filter is near-idle, so a
large gap usually means the pipeline drives a *different* QAN backend than the sweep
(see the backend caveat at the top).

In [ ]:
best  = study.best_params
g_vec = np.array([0., 0., -9.81])

for seed in range(3):
    cfg = RunConfig(
        network=NetworkConfig(sigma=best["sigma"],
                              offset_magnitude=best["offset_magnitude"]),  # spacing/alpha/b = defaults
        experiment=Arena2DConfig(n_steps=5000, seed=seed),
    )
    res = Arena2DExperiment(cfg, record=False).run_experiment(g=g_vec)
    print(f"seed {seed}: full-pipeline mean_norm_error = {res.mean_norm_error:.4f}")
    gc.collect()

## 5 — Scaling notes

- **n = 64 / rented GPU:** only after the dense `connectivity_matrix` is built lazily
  (or skipped) in the MADE `CAN` — otherwise construction OOMs before torch runs.
- **Higher dimensions:** once you add the filter params (`kappa`, `bingham_decay`) and
  seed-averaging, keep using TPE — that's where it beats a grid. With just these two
  axes a grid is equally good and more interpretable; TPE earns its keep as dims/cost grow.
- **Pruning** (ASHA/median) needs intermediate values. To enable it for expensive runs,
  have `simulate` call `trial.report(running_error, t)` at checkpoints and
  `if trial.should_prune(): raise optuna.TrialPruned()` — Optuna then kills a trial
  whose bump is already lost by, say, t = 500 instead of running it to the end.